# Group Order Exploration: C4 vs C8 vs C16

## Research Question

> **Does increasing the rotational resolution of the discrete symmetry group improve classification accuracy, and is there a point of diminishing returns?**

| Group | Order N | Rotation step | Angles covered |
|-------|:-------:|:-------------:|:--------------:|
| **C4** | 4 | 90° | 0°, 90°, 180°, 270° |
| **C8** | 8 | 45° | 0°, 45°, 90°, ..., 315° |
| **C16** | 16 | 22.5° | 0°, 22.5°, 45°, ..., 337.5° |

Higher N provides finer rotational resolution but increases computational cost and the severity of GroupPooling compression.

## Experimental Protocol

- **Architecture**: Same fiber counts (4, 8, 16, 32)
- **Optimizer**: AdamW (lr=0.001, wd=1e-4)
- **Loss**: CE | **Seed**: 42 | **Epochs**: 30 | **Patience**: 10

Only the group order N varies.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
from pathlib import Path

from src.models.factory import build_model
from src.models.equivariant_variants import EquivariantCNNVariant
from src.datasets.factory import build_dataloaders
from src.training.trainer import Trainer
from src.training.losses import build_loss
from src.training.optimizers import build_optimizer
from src.evaluation.classification import evaluate_classification
from src.evaluation.rotation import evaluate_rotation_sensitivity
from src.utils.seed import set_seed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load experiment configurations from YAML
with open('../configs/experiments.yaml') as f:
    ALL_EXPERIMENTS = yaml.safe_load(f)

BASE_CONFIG = {
    'data_root': 'data/raw/ModelNet10_views',
    'splits_file': 'data/processed/splits.json',
    'batch_size': 32,
    'epochs': 30,
    'patience': 10,
}

In [ ]:
def train_and_evaluate(exp_name, label):
    """Full training run with per-epoch history tracking."""
    exp_cfg = ALL_EXPERIMENTS[exp_name]
    config = {**BASE_CONFIG, **exp_cfg}
    
    exp_dir = Path('..') / 'outputs' / exp_name
    results_file = exp_dir / 'evaluation_results.json'
    history_file = exp_dir / 'training_history.json'
    
    if results_file.exists():
        print(f'  Results exist for {exp_name}, loading...')
        with open(results_file) as f:
            eval_results = json.load(f)
        history = None
        if history_file.exists():
            with open(history_file) as f:
                history = json.load(f)
        return eval_results.get('overall_accuracy', 0), history, eval_results
    
    set_seed(42)
    
    train_loader, val_loader = build_dataloaders(config, augment=config.get('augmentation', False))
    model = build_model(config['model']).to(device)
    optimizer = build_optimizer(model, config)
    loss_fn = build_loss(config['loss'])
    
    exp_dir.mkdir(parents=True, exist_ok=True)
    (exp_dir / 'checkpoints').mkdir(parents=True, exist_ok=True)
    (exp_dir / 'checkpoints').mkdir(parents=True, exist_ok=True)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc = 0
    patience_counter = 0
    epochs = config.get('epochs', 30)
    patience = config.get('patience')
    
    for epoch in range(epochs):
        model.train()
        train_loss, correct, total = 0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(correct / total)
        
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = loss_fn(out, y)
                val_loss += loss.item()
                correct += (out.argmax(1) == y).sum().item()
                total += y.size(0)
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(correct / total)
        
        val_acc = history['val_acc'][-1]
        print(f'  [{label}] Epoch {epoch}: train_acc={history["train_acc"][-1]:.4f}, val_acc={val_acc:.4f}')
        
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'best_val_acc': best_acc},
                       exp_dir / 'checkpoints' / 'best.pt')
        else:
            patience_counter += 1
            if patience is not None and patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break
    
    ckpt_path = exp_dir / 'checkpoints' / 'best.pt'
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
        model.load_state_dict(ckpt['model_state_dict'])
    
    report, mean_loss, _, _ = evaluate_classification(model, val_loader, device, loss_fn)
    eval_results = {
        'experiment': exp_name,
        'overall_accuracy': report.get('accuracy', best_acc),
        'overall_loss': mean_loss,
        'macro_f1': report.get('macro avg', {}).get('f1-score', 0),
    }
    
    with open(results_file, 'w') as f:
        json.dump(eval_results, f, indent=4)
    with open(history_file, 'w') as f:
        json.dump(history, f, indent=4)
    
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    return best_acc, history, eval_results

---
## Section 1: Training

| Variant | Group | Fibers | Effective channels | Compression |
|---------|:-----:|:------:|:------------------:|:-----------:|
| C4  | C4  | 4,8,16,32 | 16→128 | 4× |
| C8  | C8  | 4,8,16,32 | 32→256 | 8× |
| C16 | C16 | 4,8,16,32 | 64→512 | 16× |

In [ ]:
# Select group order experiments from YAML
group_experiments = {
    'C4':  'GROUP-C4',
    'C8':  'Eq-1',        # Reuse existing baseline
    'C16': 'GROUP-C16',
}

for label, exp_name in group_experiments.items():
    cfg = ALL_EXPERIMENTS[exp_name]
    print(f'{label}: {exp_name} → model={cfg["model"]}')

group_results = {}
group_histories = {}
for label, exp_name in group_experiments.items():
    print(f'\n=== {label} ({exp_name}) ===')
    best_acc, history, eval_res = train_and_evaluate(exp_name, label)
    group_results[label] = eval_res
    group_histories[label] = history
    print(f'  → Best: {best_acc:.4f}')    import gc
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# Parameter counts and efficiency metrics
group_model_keys = {'C4': 'model_eq_c4', 'C8': 'model_eq', 'C16': 'model_eq_c16'}
group_params = {}
for name, key in group_model_keys.items():
    model = build_model(key)
    group_params[name] = sum(p.numel() for p in model.parameters())
    del model

group_orders = {'C4': 4, 'C8': 8, 'C16': 16}
group_steps = {'C4': '90°', 'C8': '45°', 'C16': '22.5°'}
group_names = ['C4', 'C8', 'C16']

rows = []
for n in group_names:
    acc = group_results[n].get('overall_accuracy', 0)
    p = group_params[n]
    acc_per_m = (acc * 100) / (p / 1e6) if p > 0 else 0
    rows.append({
        'Group': n,
        'Order (N)': group_orders[n],
        'Step': group_steps[n],
        'Parameters': f'{p:,}',
        'Accuracy': f'{acc*100:.2f}%',
        'Macro F1': f"{group_results[n].get('macro_f1', 0):.4f}",
        'Acc/M Params': f'{acc_per_m:.1f}',
        'Compression': f'{group_orders[n]}×',
    })

group_df = pd.DataFrame(rows)
print('\n=== Group Order Study Results (Full Training) ===')
print(group_df.to_string(index=False))

In [ ]:
# Visualization: Accuracy and Parameters by Group Order
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_group = {'C4': '#e74c3c', 'C8': '#2ecc71', 'C16': '#3498db'}
accs = [group_results[n].get('overall_accuracy', 0) for n in group_names]
params_list = [group_params[n] for n in group_names]

for i, (name, acc) in enumerate(zip(group_names, accs)):
    ax1.bar(i, acc, color=colors_group[name], edgecolor='black', linewidth=0.8, width=0.6)
    ax1.text(i, acc + 0.005, f'{acc*100:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax1.set_xticks(range(len(group_names)))
ax1.set_xticklabels([f'{n}\n(step={group_steps[n]})' for n in group_names], fontsize=10)
ax1.set_ylabel('Validation Accuracy', fontsize=12)
ax1.set_title('Accuracy vs Group Order', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

for i, (name, p) in enumerate(zip(group_names, params_list)):
    ax2.bar(i, p, color=colors_group[name], edgecolor='black', linewidth=0.8, width=0.6)
    ax2.text(i, p + 500, f'{p:,}', ha='center', fontsize=9, fontweight='bold')

ax2.set_xticks(range(len(group_names)))
ax2.set_xticklabels([f'{n}\n(N={group_orders[n]})' for n in group_names], fontsize=10)
ax2.set_ylabel('Total Parameters', fontsize=12)
ax2.set_title('Parameter Count vs Group Order', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig('../outputs/figures/group_order_study.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Efficiency scatter: Accuracy vs Parameters
fig, ax = plt.subplots(figsize=(10, 6))

for name in group_names:
    acc = group_results[name].get('overall_accuracy', 0)
    p = group_params[name]
    ax.scatter(p, acc, s=200, c=colors_group[name], edgecolors='black', linewidths=1, zorder=5)
    ax.annotate(f'{name} (N={group_orders[name]})', (p, acc), textcoords='offset points',
              xytext=(12, 8), fontsize=11, fontweight='bold')

ax.plot(params_list, accs, '--', color='gray', alpha=0.5, linewidth=1.5)

ax.set_xlabel('Total Parameters', fontsize=12)
ax.set_ylabel('Validation Accuracy', fontsize=12)
ax.set_title('Accuracy vs Parameters: Effect of Group Order', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/group_order_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2: Training Curves

In [ ]:
# Training curves
models_with_history = {k: v for k, v in group_histories.items() if v is not None}

if models_with_history:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    for name, hist in models_with_history.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        color = colors_group.get(name, '#333333')
        axes[0].plot(epochs, hist['train_loss'], '-', color=color, label=name, linewidth=1.8)
        axes[1].plot(epochs, hist['val_loss'], '-', color=color, label=name, linewidth=1.8)
        axes[2].plot(epochs, hist['val_acc'], '-', color=color, label=name, linewidth=1.8)
    
    axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)
    
    axes[1].set_title('Validation Loss', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)
    
    axes[2].set_title('Validation Accuracy', fontsize=13, fontweight='bold')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
    axes[2].legend(fontsize=10); axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/group_order_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training history available. Re-run training to generate curves.')

---
## Section 3: Feature Dimension Comparison

In [ ]:
# Feature dimensions across group orders
x_dummy = torch.randn(1, 3, 224, 224)

group_configs = {
    'C4 (N=4)':   dict(N=4,  fibers=(4, 8, 16, 32)),
    'C8 (N=8)':   dict(N=8,  fibers=(4, 8, 16, 32)),
    'C16 (N=16)': dict(N=16, fibers=(4, 8, 16, 32)),
}

print('=== Feature Dimensions by Group Order ===')
print(f'{"Stage":<25} {"C4":>8} {"C8":>8} {"C16":>8}')
print('-' * 55)

group_infos = {}
for name, kwargs in group_configs.items():
    model = EquivariantCNNVariant(**kwargs)
    group_infos[name] = model.get_feature_info(x_dummy)
    del model

stages = [
    ('Input', 'input', 'channels'),
    ('After Block 1', 'after_block1', 'equivariant_channels'),
    ('After Block 2', 'after_block2', 'equivariant_channels'),
    ('After Block 3', 'after_block3', 'equivariant_channels'),
    ('Before GroupPool', 'before_gpool', 'equivariant_channels'),
    ('After GroupPool', 'after_gpool', 'invariant_channels'),
    ('Classifier Input', 'after_spatial_pool', 'features'),
]

for stage_name, key, ch_key in stages:
    vals = [group_infos[n][key][ch_key] for n in group_configs.keys()]
    print(f'{stage_name:<25} {vals[0]:>8} {vals[1]:>8} {vals[2]:>8}')

print()
print('Key observation: GroupPooling always outputs 32 channels (= fiber count),')
print('regardless of group order. But the compression ratio increases with N.')
print(f'Compression: C4=4×, C8=8×, C16=16×')

---
## Section 4: Analysis & Conclusions

### Does higher rotational resolution help?

1. **C4 → C8 → C16**: Each step doubles the number of discrete rotations the model is equivariant to.

2. **Parameter scaling**: With fixed fiber counts, parameters scale linearly with N.

3. **Compression trade-off**: Higher N means more aggressive GroupPooling compression (C16 discards 16× the information that C4 does).

### Is C8 sufficient?

C8 captures 45° rotation steps, already fine-grained relative to the 30° render intervals in ModelNet10. The marginal benefit of C16 (22.5° steps) must be weighed against the 2× parameter increase and 2× compression increase.

### Trade-off Summary

Increasing rotational resolution introduces competing effects:
- **Positive**: Better approximation of continuous rotation symmetry
- **Negative**: More aggressive GroupPooling compression, higher computational cost

The optimal group order balances rotational fidelity against information preservation.